# 03 · Tracking y unión de fragmentos (S3 → S5)

**Historias:** US-02.4 (tracker), US-02.5 (trayectoria por visita), US-01.6 (prueba virgen). **Kernel:** `colibri-yolo`.

| Sección | Qué reproduce | Decisión | Cifra de referencia |
|---|---|---|---|
| 1 | Videos `test_` y GT v1 / v2 | D-07 | 216 frames GT en 4 videos |
| 2 | 5 configuraciones de tracker en `test_` | **D-09** ByteTrack combo | `resultados/S5_tracking_union_test/gt*/metricas_tracking_test.csv` |
| 3 | Rejilla de unión en vv + 4 `train_` | (elegida 0,5 s × 4, luego descartada) | `union_rejilla.csv` |
| 4 | Actual / elegida / prudente en `test_` con GT v1 y v2 | **D-10** prudente 1 s × 2,5 | `union_test_gt1_gt2.csv`: IDF1 visita 0,667, 0 puentes falsos |
| 5 | Trayectorias finales de `test_` | entrada de 04 y 06 | `trayectorias_prudente/*.csv` |

**Modo `artefactos`** (por defecto): usa las trayectorias detalladas congeladas de S5 (`datos_derivados/tracks/`),
recalcula todas las métricas y las compara. Tarda unos minutos (solo lee videos para los cortes de escena y las figuras).
**Modo `recalcular`**: vuelve a correr el tracking de las 5 configuraciones en GPU (≈ 20 pasadas de video).

Las salidas nuevas van a `runs/03/`; las de referencia en `resultados/` no se tocan.

In [ ]:
%run 00_comun.ipynb
OUT = RUNS / "03"; OUT.mkdir(parents=True, exist_ok=True)
REF = P["resultados"] / "S5_tracking_union_test"

## 1. Videos de prueba y verdad de terreno

Los cuatro `test_` nunca han entrado en entrenamiento ni en validación (D-07). GT v1 = etiquetas de la versión 1 de
`humming`; GT v2 = la versión corregida tras revisar los errores de S5 (`datos_derivados/humming/cambios_v1_v2.csv`).
Un frame con 2+ cajas no forma ni separa visitas.

In [ ]:
VIDS_TEST = videos(("test",))
GT = {v: cargar_gt_test(v, VIDS_TEST) for v in ("v1", "v2")}
for v, g in GT.items():
    print(f"GT {v}: {len(g)} frames | {int(g.n_gt.sum())} cajas | máx. aves por frame {int(g.n_gt.max())}")
display(pd.concat({v: g.groupby("video_id").agg(frames=("frame", "size"), cajas=("n_gt", "sum")) for v, g in GT.items()}, axis=1)
        .rename(index=corto))

## 2. Cinco configuraciones de tracker (S5, D-09)

Mismas funciones que S3 (`evaluate_tracks`, `build_trajectory`), solo cambia la GT. Criterio fijado **antes** de ver
resultados: mínimo `ids_por_ave + IDSW`, luego mayor IDF1, luego mayor MOTA; ByteTrack si empatan.
La configuración `e_viejo_oc_combo` (modelo anterior) solo es referencia.

In [ ]:
def metricas_tracking(GT, vids):
    """Cuerpo de main() de tracking_test.py: métricas por video y GLOBAL para cada configuración."""
    rows, short_all, TR = [], [], {}
    for cfg_name, (weights, tracker, over) in CONFIGS.items():
        per_vid = []
        for vid, info in vids.items():
            tr = get_tracks(vid, info, cfg_name); TR[(cfg_name, vid)] = tr
            m, per = evaluate_tracks(GT, vid, tr)
            traj, tinfo = build_trajectory(tr)
            st = short_tracks(tr, GT[GT.video_id == vid])
            m.update({"config": cfg_name, "segmentos": int(traj.segmento.nunique()) if len(traj) else 0,
                      "frames_traj": len(traj), "tracks_cortos": len(st)})
            per_vid.append(m); short_all.append(st.assign(config=cfg_name, video_id=vid))
        pv = pd.DataFrame(per_vid)
        n_gt, tp, fp = pv.n_gt.sum(), pv.TP.sum(), pv.FP.sum()
        rows += per_vid
        rows.append({"config": cfg_name, "video_id": "GLOBAL", "tracker": tracker, "n_gt": int(n_gt), "TP": int(tp),
                     "FN": int(n_gt - tp), "FP": int(fp), "IDSW": int(pv.IDSW.sum()),
                     "ids_por_ave": int(pv.ids_por_ave.sum()), "ids_totales": int(pv.ids_totales.sum()),
                     "recall": tp / n_gt, "precision": tp / max(tp + fp, 1),
                     "MOTA": 1 - (n_gt - tp + fp + pv.IDSW.sum()) / n_gt, "IDF1": 2 * pv.idtp.sum() / (n_gt + tp + fp),
                     "IoU_medio": float(np.average(pv.IoU_medio.fillna(0), weights=pv.TP)) if tp else np.nan,
                     "segmentos": int(pv.segmentos.sum()), "frames_traj": int(pv.frames_traj.sum()),
                     "tracks_cortos": int(pv.tracks_cortos.sum()), "idtp": int(pv.idtp.sum())})
    cols = ["config", "video_id", "tracker", "n_gt", "TP", "FN", "FP", "IDSW", "ids_por_ave", "ids_totales",
            "tracks_cortos", "recall", "precision", "MOTA", "IDF1", "IoU_medio", "segmentos", "frames_traj", "fps_proceso"]
    return pd.DataFrame(rows)[cols], pd.concat(short_all, ignore_index=True), TR

def elegir_tracker(M):
    G = M[(M.video_id == "GLOBAL") & M.config.isin(DECISION)].copy()
    G["_k1"] = G.ids_por_ave + G.IDSW
    G = G.sort_values(["_k1", "IDF1", "MOTA"], ascending=[True, False, False])
    best = tuple(G.iloc[0][["_k1", "IDF1", "MOTA"]].astype(float).round(3))
    tied = [c for c in G.config if tuple(G[G.config == c].iloc[0][["_k1", "IDF1", "MOTA"]].astype(float).round(3)) == best]
    bt = [c for c in tied if "_bt_" in c]
    return (bt[0] if bt else G.config.iloc[0]), tied, G

M, CORTOS, TR = {}, {}, {}
for v in ("v1", "v2"):
    M[v], CORTOS[v], TR = metricas_tracking(GT[v], VIDS_TEST)
    (OUT / f"gt{v[1]}").mkdir(exist_ok=True); M[v].to_csv(OUT / f"gt{v[1]}" / "metricas_tracking_test.csv", index=False)
    elegida, empate, G = elegir_tracker(M[v])
    print(f"GT {v}: elegida {elegida}", "(empate: " + ", ".join(empate) + ")" if len(empate) > 1 else "")
    display(G.drop(columns="_k1").round(3))
    comparar(M[v], REF / f"gt{v[1]}" / "metricas_tracking_test.csv", ["config", "video_id"],
             ["TP", "FP", "IDSW", "ids_por_ave", "recall", "MOTA", "IDF1"], nombre=f"tracking GT {v} frente a S5")
assert elegir_tracker(M["v1"])[0] == CFG["vigente"]["tracker"], "la elección ya no coincide con D-09"

## 3. Rejilla de unión en vv + 4 `train_` (S5)

El ajuste se hace en videos que **sí** vio el detector (vv y cuatro `train_` de YouTube), para no gastar la prueba.
Hueco de 0,25 a 5 s × distancia de 1 a 4 veces el lado medio de la caja. Criterio fijado antes de mirar: mínimos
puentes falsos, luego mayor IDF1 de visita, luego menos segmentos por visita.

La rejilla eligió 0,5 s × 4; en `test_` dibujaba rectas largas en J4PK, así que Yilber eligió la **prudente**
(1 s × 2,5) con la GT v2 (sección 4).

In [ ]:
# Portado de video-val/resultados/tracking_v2/union/union.py
GRID = [(g, d) for g in (0.25, 0.5, 1, 2, 3, 5) for d in (1, 1.5, 2.5, 4)]

# Portado de video-val/resultados/tracking_v2/union/union.py
def evaluate(vids, GT, configs, rol, extra=None):
    rows = []
    for vid, v in vids.items():
        if v["rol"] not in rol:
            continue
        tr = get_tracks(vid, v); cuts = scene_cuts(v["path"]); g = GT[GT.video_id == vid]
        for name, fn in configs.items():
            traj = fn(tr, cuts)
            m = visit_metrics(traj, g)
            rows.append(dict(config=name, video_id=vid, rol=v["rol"], cortes=len(cuts), **m))
            if extra:
                gv = g[g.split == "valid"]
                if len(gv):
                    mv = visit_metrics(traj, gv)
                    rows[-1].update(cobertura_valid=mv["cobertura"], cajas_valid=mv["cajas"])
                extra(name, vid, v, traj, g)
    return pd.DataFrame(rows)

VIDS_AJUSTE = videos(("vv", "train"), solo_ajuste_union=True)
GT_U = cargar_gt_union({**VIDS_AJUSTE, **VIDS_TEST})
UCONF = {"actual_30f_1.5": lambda tr, cuts: union_reference(tr)}
for gs, dm in GRID:
    UCONF[f"g{gs}s_d{dm}"] = (lambda gs, dm: lambda tr, cuts: union_tuned(tr, gs, dm, cuts))(gs, dm)

R = evaluate(VIDS_AJUSTE, GT_U, UCONF, {"vv", "train"}, extra=lambda *a: None)
R.to_csv(OUT / "union_rejilla_por_video.csv", index=False)
A = pd.DataFrame([dict(config=c, **aggregate(g.to_dict("records"))) for c, g in R.groupby("config", sort=False)])
A = A.sort_values(["puentes_falsos", "IDF1_visita", "seg_por_visita"], ascending=[True, False, True]).reset_index(drop=True)
A.to_csv(OUT / "union_rejilla.csv", index=False)
print("ELEGIDA por la rejilla:", A[A.config != "actual_30f_1.5"].iloc[0].config)
display(A.head(8).round(3))
comparar(A, REF / "union_rejilla.csv", ["config"], ["puentes_falsos", "IDF1_visita", "seg_por_visita", "cobertura"],
         nombre="rejilla de unión frente a S5")

## 4. Confirmación en `test_` con GT v1 y GT v2 (D-10)

Una sola pasada con las tres uniones candidatas. Criterio: mínimos puentes falsos, luego mayor IDF1 de visita,
luego menos puentes largos.

In [ ]:
# Portado de video-val/resultados/tracking_v2/eval_gt2/union_gt2.py
UNION_TEST = {
    "actual_30f_1.5": lambda tr, cuts: union_reference(tr),
    "elegida_0.5s_4": lambda tr, cuts: union_tuned(tr, 0.5, 4, cuts),
    "prudente_1s_2.5": lambda tr, cuts: union_tuned(tr, 1, 2.5, cuts),
}

rows, br_rows = [], []
for gt_name, g_all in (("GT_v1", GT["v1"]), ("GT_v2", GT["v2"])):
    for vid, v in VIDS_TEST.items():
        tr = get_tracks(vid, v); cuts = scene_cuts(v["path"]); g = g_all[g_all.video_id == vid]
        for name, fn in UNION_TEST.items():
            traj = fn(tr, cuts)
            m = visit_metrics(traj, g)
            br, n_long = bridges(traj, v["W"], v["H"])
            rows.append(dict(gt=gt_name, config=name, video_id=vid, **m, puentes=len(br), puentes_largos=n_long))
            if gt_name == "GT_v2":
                br_rows += [dict(config=name, video_id=vid, **b, largo=b["long_px"] > min(v["W"], v["H"]) / 4) for b in br]
RT = pd.DataFrame(rows)
agg = [dict(gt=gt, config=c, video_id="GLOBAL", **aggregate(g.to_dict("records")), puentes=int(g.puentes.sum()),
            puentes_largos=int(g.puentes_largos.sum())) for (gt, c), g in RT.groupby(["gt", "config"], sort=False)]
RT = pd.concat([RT, pd.DataFrame(agg)], ignore_index=True)
RT.to_csv(OUT / "union_test_gt1_gt2.csv", index=False); pd.DataFrame(br_rows).to_csv(OUT / "puentes_gt2.csv", index=False)
G2 = RT[(RT["gt"] == "GT_v2") & (RT.video_id == "GLOBAL")].sort_values(["puentes_falsos", "IDF1_visita", "puentes_largos"],
                                                                        ascending=[True, False, True])
print("GANADORA (GT v2):", G2.config.iloc[0])
display(RT[RT.video_id == "GLOBAL"][["gt", "config", "visitas", "puentes_falsos", "IDF1_visita", "seg_por_visita",
                                      "cobertura", "segmentos", "puentes", "puentes_largos"]].round(3))
comparar(RT, REF / "union_test_gt1_gt2.csv", ["gt", "config", "video_id"],
         ["puentes_falsos", "IDF1_visita", "segmentos", "puentes_largos"], nombre="unión en test_ frente a S5")

## 5. Trayectorias finales (entrada de los notebooks 04 y 06)

Unión prudente sobre ByteTrack combo, con la columna `es_puente`. Se escriben en `runs/03/trayectorias_prudente/`
y se comparan fila a fila con las de S5 (`resultados/S5_tracking_union_test/trayectorias_prudente/`), que son las que
leen 04 y 06 en modo `artefactos`.

In [ ]:
fin = OUT / "trayectorias_prudente"; fin.mkdir(exist_ok=True)
for vid, v in VIDS_TEST.items():
    tr = get_tracks(vid, v); cuts = scene_cuts(v["path"]); g = GT["v2"][GT["v2"].video_id == vid]
    traj = union_vigente(tr, cuts)
    traj.to_csv(fin / f"trayectoria_{vid}.csv", index=False)
    m = visit_metrics(traj, g); _, n_long = bridges(traj, v["W"], v["H"])
    draw(v["path"], traj, g, f"{corto(vid)} | prudente_1s_2.5 | seg {m['segmentos']} | visitas {m['visitas']} | "
         f"puentes falsos {m['puentes_falsos']} | largos {n_long} | IDF1v {m['IDF1_visita']:.2f}", fin / f"trayectoria_{vid}.png")
    ref = REF / "trayectorias_prudente" / f"trayectoria_{vid}.csv"
    if ref.exists():
        comparar(traj, ref, ["frame"], ["cx_px", "cy_px", "segment_id"], tol=TOL["px"], nombre=f"trayectoria {corto(vid)}")

## Conclusiones (S5)

- **Tracker (D-09):** ByteTrack combo (`track_buffer` 300, `match_thresh` 0,95, `new_track_thresh` 0,5) con el
  detector `yolo26n_ft_v2`. Gana por el criterio fijado y por desempate a favor de ByteTrack.
- **Unión (D-10):** prudente, 1 s y 2,5×. Con la GT v2: IDF1 de visita 0,667, **0 puentes falsos** y 3 puentes largos.
  La elegida por la rejilla (0,5 s × 4) dibujaba rectas largas en J4PK.
- **Incremento:** trayectorias por visita de los 4 videos de prueba, con los puentes marcados, listas para 04.
- **Límite:** la prueba son 4 videos y 216 frames etiquetados; cada cifra es de un solo ave por video.